### Q3: Did districts where housing problems grew fastest from 2022 to 2025 also see serious crashes grow?

This question checks whether worsening housing conditions and worsening road safety tend to happen in the same community districts over time. Instead of looking at which district has the most problems, we look at which districts **changed** the most.

In [ ]:
def build_q3(df_integrated):
    """Q3: Did districts where housing problems grew fastest also see serious crashes grow?"""
    # Use the first and last year in the data (normally 2022 and 2025)
    first_year, last_year = df_integrated['year'].min(), df_integrated['year'].max()
    if first_year == last_year:
        return px.scatter(title='Select at least two years to see growth')
    d = df_integrated[df_integrated['year'].isin([first_year, last_year])]
    # One row per district, with a column for each measure in each year
    p = d.pivot_table(index=['cd_label', 'borough'], columns='year',
                      values=['housing_problems', 'crashes'], aggfunc='sum').reset_index()
    p.columns = [f'{a}_{b}' if b != '' else a for a, b in p.columns]
    # Skip districts with 0 in the first year (can't divide by 0)
    p = p[(p[f'housing_problems_{first_year}'] > 0) & (p[f'crashes_{first_year}'] > 0)]
    p['housing_change_%'] = (p[f'housing_problems_{last_year}'] - p[f'housing_problems_{first_year}']) / p[f'housing_problems_{first_year}'] * 100
    p['crash_change_%'] = (p[f'crashes_{last_year}'] - p[f'crashes_{first_year}']) / p[f'crashes_{first_year}'] * 100
    fig = px.scatter(p, x='housing_change_%', y='crash_change_%', color='borough',
                     hover_name='cd_label',
                     hover_data={f'housing_problems_{first_year}': True, f'housing_problems_{last_year}': True,
                                 f'crashes_{first_year}': True, f'crashes_{last_year}': True},
                     title=f'Growth vs growth: housing problems vs serious crashes, {first_year}–{last_year}',
                     labels={'housing_change_%': 'Change in housing problems (%)',
                             'crash_change_%': 'Change in serious crashes (%)'})
    fig.add_vline(x=0, line_dash='dash', line_color='grey')
    fig.add_hline(y=0, line_dash='dash', line_color='grey')
    return fig

build_q3(df_integrated).show()

In [ ]:
# Helper: numbers for the Q3 answer (NOT part of the website)
d = df_integrated[df_integrated['year'].isin([2022, 2025])]
p = d.pivot_table(index=['cd_label', 'borough'], columns='year',
                  values=['housing_problems', 'crashes'], aggfunc='sum')
h = (p[('housing_problems', 2025)] - p[('housing_problems', 2022)]) / p[('housing_problems', 2022)] * 100
c = (p[('crashes', 2025)] - p[('crashes', 2022)]) / p[('crashes', 2022)] * 100
q = pd.DataFrame({'housing_%': h.round(1), 'crash_%': c.round(1)}).reset_index()

print('Districts:', len(q))
print('Both up:', ((q['housing_%'] > 0) & (q['crash_%'] > 0)).sum())
print('Both down:', ((q['housing_%'] < 0) & (q['crash_%'] < 0)).sum())
print('Housing up, crashes down:', ((q['housing_%'] > 0) & (q['crash_%'] < 0)).sum())
print('Housing down, crashes up:', ((q['housing_%'] < 0) & (q['crash_%'] > 0)).sum())
print('Correlation r =', round(q['housing_%'].corr(q['crash_%']), 2))
print('Median housing change %:', q['housing_%'].median(), '| Median crash change %:', q['crash_%'].median())
print('\nTop 5 where BOTH grew the most:')
print(q[(q['housing_%'] > 0) & (q['crash_%'] > 0)].sort_values('crash_%', ascending=False).head(5).to_string(index=False))
print('\nCrash counts 2022 vs 2025 (to show small numbers):')
print('Average crashes per district in 2022:', round(p[('crashes', 2022)].mean(), 1))

Districts: 59
Both up: 29
Both down: 2
Housing up, crashes down: 22
Housing down, crashes up: 5
Correlation r = -0.09
Median housing change %: 16.5 | Median crash change %: 7.0

Top 5 where BOTH grew the most:
      cd_label  borough  housing_%  crash_%
   Queens CD 7   QUEENS       33.7    132.7
Brooklyn CD 13 BROOKLYN       27.0     81.2
   Queens CD 8   QUEENS       45.3     80.4
 Brooklyn CD 1 BROOKLYN        3.1     56.9
   Bronx CD 11    BRONX       13.8     56.0

Crash counts 2022 vs 2025 (to show small numbers):
Average crashes per district in 2022: 43.9


**Answer:** Housing problems rose in 51 of the 59 community districts between 2022 and 2025 (median change +16.5%), but serious crashes did not follow the same pattern. Only 29 districts saw both measures grow, while 22 saw housing problems rise and crashes fall, and the correlation between the two growth rates is close to zero (r = −0.09). The districts where both grew the most were Queens CD 7 (housing +33.7%, crashes +132.7%), Brooklyn CD 13 (+27.0%, +81.2%) and Queens CD 8 (+45.3%, +80.4%). So faster growth in housing problems is **not associated** with faster growth in serious crashes; the two trends appear to move independently.

**Limitation:** Serious crashes are rare (about 44 per district in 2022), so a small change in count produces a large percentage change, which makes the crash growth rates noisy. Comparing only two years (2022 and 2025) also ignores what happened in between.


### Q11: Which streets are NYC's most dangerous corridors, and are they getting worse?

Serious crashes are not spread evenly across the city; they tend to cluster on certain streets. Finding the streets with the most serious crashes, and checking whether their numbers are rising or falling each year, shows where street-safety work would matter most.

In [ ]:
def build_q11(df_crashes):
    """Q11: Which streets are NYC's most dangerous corridors, and are they getting worse?"""
    d = df_crashes.dropna(subset=['on_street']).copy()   # about 19% of crashes have no street name
    # Clean street names so the same street is not counted as different streets
    s = d['on_street'].str.upper().str.strip().str.replace(r'\s+', ' ', regex=True)
    s = s.str.replace(r'\bAVE\b', 'AVENUE', regex=True)
    s = s.str.replace(r'\bBLVD\b', 'BOULEVARD', regex=True)
    s = s.str.replace(r'\bST$', 'STREET', regex=True)     # only at the end, so "ST NICHOLAS" stays as is
    s = s.str.replace(r'\bPKWY\b', 'PARKWAY', regex=True)
    s = s.str.replace(r'\bEXPY\b', 'EXPRESSWAY', regex=True)
    d['street'] = s
    # Top 10 streets by total serious crashes, counted per year
    top10 = d['street'].value_counts().head(10).index
    counts = (d[d['street'].isin(top10)]
              .groupby(['street', 'year']).size().reset_index(name='crashes'))
    counts['year'] = counts['year'].astype(str)   # one distinct colour per year
    fig = px.bar(counts, x='crashes', y='street', color='year', orientation='h',
                 title='Most dangerous corridors: serious crashes on the top 10 streets, by year',
                 labels={'crashes': 'Serious crashes', 'street': 'Street', 'year': 'Year'},
                 category_orders={'year': sorted(counts['year'].unique())})
    fig.update_layout(yaxis={'categoryorder': 'total ascending'})   # most dangerous street on top
    return fig

build_q11(df_crashes).show()

In [ ]:
# Helper: numbers for the Q11 answer (NOT part of the website)
fig = build_q11(df_crashes)
t = pd.DataFrame([(tr.name, s, x) for tr in fig.data for s, x in zip(tr.y, tr.x)],
                 columns=['year', 'street', 'crashes'])
table = t.pivot_table(index='street', columns='year', values='crashes', aggfunc='sum').fillna(0).astype(int)
table['total'] = table.sum(axis=1)
table['change_22_to_25'] = table['2025'] - table['2022']
print(table.sort_values('total', ascending=False))
print('\nCrashes with no street name: ', round(df_crashes['on_street'].isna().mean() * 100, 1), '%')

year                     2022  2023  2024  2025  total  change_22_to_25
street                                                                 
BROADWAY                   30    46    40    39    155                9
BELT PARKWAY               34    30    34    38    136                4
GRAND CENTRAL PARKWAY      23    21    30    22     96               -1
MAJOR DEEGAN EXPRESSWAY    16    29    19    27     91               11
3 AVENUE                   21    27    23    19     90               -2
FDR DRIVE                  20    23    22    19     84               -1
LONG ISLAND EXPRESSWAY     15    17    23    25     80               10
CROSS ISLAND PARKWAY       17    14    28    17     76                0
FLATBUSH AVENUE            20    19    20    17     76               -3
NORTHERN BOULEVARD         21    22    18    12     73               -9

Crashes with no street name:  19.1 %


**Answer:** Broadway is NYC's most dangerous corridor, with 155 serious crashes between 2022 and 2025, followed by Belt Parkway (136) and Grand Central Parkway (96). Six of the top 10 corridors are highways (Belt Parkway, Grand Central Parkway, Major Deegan Expressway, FDR Drive, Long Island Expressway and Cross Island Parkway), showing that serious crashes are concentrated on high-speed roads as well as busy city streets. Some corridors tend to be getting worse: the Major Deegan Expressway rose from 16 serious crashes in 2022 to 27 in 2025, the Long Island Expressway from 15 to 25, and Broadway from 30 to 39. Others improved, most clearly Northern Boulevard, which fell from 21 to 12.

**Limitation:** Long roads such as Broadway and Belt Parkway naturally collect more crashes simply because they are longer, so these are totals, not crashes per kilometre. Also, 19.1% of serious crashes have no street name and are left out, and yearly counts per street are small, so a change of a few crashes can be random.